## 대규모 AI 시스템의 병목(Bottleneck) 분석 실험

### 목표
데이터 파이프라인과 배치 처리 관점에서 AI 시스템의 성능 병목을 식별하고 개선 방법을 탐색합니다.

### 고정 변인
- **데이터셋**: Flickr8k (이미지 + 캡션) - 시뮬레이션 데이터
- **전처리 시간**: 5초/배치 (time.sleep 시뮬레이션)
- **모델 추론 시간**: 10초/배치 (time.sleep 시뮬레이션)
- **배치 크기**: 16 (고정)

### 실험 순서
1. 데이터 준비 및 탐색
2. 전처리 파이프라인 정의
3. Iter 방식 데이터 호출 (Sequential)
4. Batch 방식 데이터 호출 (Batched)
5. 모델 추론 시뮬레이션
6. 병목 분석 및 Throughput 계산
7. 개선점 탐색 및 적용

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from threading import Thread
import queue
import warnings
warnings.filterwarnings('ignore')

print("✅ 필요한 라이브러리 로드 완료")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

## 1️⃣ 데이터 준비 및 탐색 (Data Preparation)

In [ ]:
# 시뮬레이션 Flickr8k 데이터셋 생성
data_dir = 'data/flickr8k'
image_folder = os.path.join(data_dir, 'Flicker8k_Dataset')
captions_file = os.path.join(data_dir, 'Flickr8k.token.txt')
os.makedirs(image_folder, exist_ok=True)

print("=" * 80)
print("📥 시뮬레이션 Flickr8k 데이터셋 생성")
print("=" * 80)

NUM_IMAGES = 100
np.random.seed(42)

# 이미지 데이터 생성
print(f"\n🖼️  {NUM_IMAGES}개의 이미지 생성 중...")
image_ids = []
for i in range(NUM_IMAGES):
    img_array = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
    img = Image.fromarray(img_array)
    img_name = f"{i:06d}.jpg"
    img_path = os.path.join(image_folder, img_name)
    img.save(img_path)
    image_ids.append(i)
    if (i + 1) % 50 == 0:
        print(f"  ✓ {i + 1}/{NUM_IMAGES} 이미지 생성 완료")

print(f"✅ 총 {len(os.listdir(image_folder))}개의 이미지 생성 완료")

# 캡션 데이터 생성  
print(f"\n📝 캡션 데이터 생성 중...")
captions_list = [
    "a dog playing in the park",
    "a cat sitting on a bench",
    "children running on the beach",
    "a sunset over the ocean",
    "a bird flying in the sky",
]

with open(captions_file, 'w') as f:
    for idx in image_ids:
        img_id = f"{idx:06d}"
        for cap_idx in range(5):
            caption = np.random.choice(captions_list)
            f.write(f"{img_id}.jpg#{cap_idx}\t{caption}\n")

print(f"✅ 캡션 파일 생성 완료")

# 데이터 통계
captions_df = []
with open(captions_file) as f:
    for line in f:
        img_id, caption = line.strip().split('\t', 1)
        captions_df.append({'image_id': img_id, 'caption': caption})

print(f"\n📊 데이터셋 통계:")
print(f"  - 이미지 개수: {NUM_IMAGES}")
print(f"  - 캡션 개수: {len(captions_df)}")
print(f"  - 이미지당 캡션: {len(captions_df) // NUM_IMAGES}")
print("=" * 80)

## 2️⃣ 전처리 파이프라인 정의 (Preprocessing Pipeline)

In [ ]:
# 전처리 파이프라인
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class Flickr8kDataset(Dataset):
    """Flickr8k 이미지-캡션 데이터셋 with 시뮬레이션"""
    def __init__(self, image_folder, captions_file, transform=None, simulate_time=True):
        self.image_folder = image_folder
        self.transform = transform
        self.simulate_time = simulate_time
        
        self.image_ids = []
        self.captions = []
        with open(captions_file) as f:
            for line in f:
                img_id, caption = line.strip().split('\t', 1)
                self.image_ids.append(img_id)
                self.captions.append(caption)
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        # 전처리 시뮬레이션 (5초)
        if self.simulate_time:
            time.sleep(5)
        
        img_path = os.path.join(self.image_folder, self.image_ids[idx])
        image = Image.open(img_path).convert('RGB')
        caption = self.captions[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, caption

print("=" * 80)
print("⚙️  전처리 파이프라인 설정")
print("=" * 80)
print(f"\n✅ 설정:")
print(f"  - 이미지 크기: 224x224")
print(f"  - 정규화 Mean: {IMAGENET_MEAN}")
print(f"  - 정규화 Std: {IMAGENET_STD}")
print(f"  - 전처리 시뮬레이션 시간: 5초/샘플")
print(f"  - 모델 추론 시축레이션 시간: 10초/배치")
print("=" * 80)

## 3️⃣ Sequential 파이프라인 (순차 처리)

In [ ]:
print("\n" + "=" * 80)
print("🔄 Sequential 파이프라인 (전처리 → 모델)")
print("=" * 80)

dataset = Flickr8kDataset(image_folder, captions_file, transform=transform, simulate_time=True)
BATCH_SIZE = 16
NUM_TEST_BATCHES = 2

print(f"\n설정: {NUM_TEST_BATCHES} 배치, 배치크기={BATCH_SIZE}, 총 샘플={NUM_TEST_BATCHES*BATCH_SIZE}")
print(f"예상 시간: ({NUM_TEST_BATCHES*BATCH_SIZE} 샘플 * 5초) + (10초 모델) = {NUM_TEST_BATCHES*BATCH_SIZE*5 + 10}초")

seq_times = []
print(f"\n⏱️  실행 중...") 

for batch_idx in range(NUM_TEST_BATCHES):
    batch_start = time.time()
    print(f"  [배치 {batch_idx+1}] 전처리 중...")
    for i in range(BATCH_SIZE):
        idx = batch_idx * BATCH_SIZE + i
        if idx < len(dataset):
            _ = dataset[idx]
    print(f"  [배치 {batch_idx+1}] 모델 추론 중...")
    time.sleep(10)
    batch_elapsed = time.time() - batch_start
    seq_times.append(batch_elapsed)
    print(f"  ✓ 배치 완료: {batch_elapsed:.2f}초")

total_seq_time = np.sum(seq_times)
print(f"\n📊 Sequential 파이프라인 성능:")
print(f"  - 총 시간: {total_seq_time:.2f}초")
print(f"  - 배치당 평균: {np.mean(seq_times):.2f}초")
print(f"  - 처리량: { (NUM_TEST_BATCHES * BATCH_SIZE) / total_seq_time:.2f} samples/sec")
print("=" * 80)

## 📊 병목 분석 결론 (Bottleneck Analysis Summary)

### 🔍 핵심 발견사항

#### 1. **병목 지점 식별**
데이터 파이프라인에서 발생하는 주요 병목은:

```
Sequential 파이프라인:
┌─────────────────┬────────────────────────────────┐
│ 전처리(5초 × n) │ 모델(10초)   │ → 총시간 = 5n + 10
└─────────────────┴────────────────────────────────┘
      대기 포함

Pipelined 파이프라인:
┌──────────────────────────────────────────────────────┐
│ 전처리(5초 × n) & 모델(10초)   │ → 총시간 = max(5n, 10)
│        병렬 처리 (동시)        │
└──────────────────────────────────────────────────────┘
```

#### 2. **성능 개선 효과**

| 파이프라인 | 32샘플 기준 | 개선율 |
|-----------|-----------|------|
| Sequential | 160 + 10 = 170초 | - |
| Pipelined | max(160, 10) = 160초 | 6% |
| 모델최적화 (20%) | max(160, 8) = 160초 | 6% |
| 전체 최적화 | max(96, 8) = 96초 | 43% |

#### 3. **최적화 우선순위**

🥇 **1순위: 모델 최적화**
- 모델이 가장 오래 걸림 (10초/배치)
- 1초 단축 → 최대 3.2% 개선
- 영향도: **최고**

🥈 **2순위: 파이프라이닝**
- 데이터로드 중 모델 실행
- 구현 난이도: 중간
- 영향도: 5-10%

🥉 **3순위: 배치 크기 증가**
- 메모리 허용 범위에서 증가
- 1회당 처리량 증가
- 영향도: 상황 의존

### 💡 실제 적용 권장사항

**즉시 적용 (간단)**
- ✅ 파이프라이닝/멀티프로세싱 도입
- ✅ 캐싱/프리페칭 구현
- 효과: ~10% 개선

**중기 적용 (복잡)**
- ✅ 모델 량자화 (quantization)
- ✅ 지식 증류 (knowledge distillation)
- ✅ 배치 크기 조정
- 효과: ~20-30% 개선

**장기 적용 (인프라)**
- ✅ GPU/TPU 업그레이드
- ✅ 분산 처리 (distributed training)
- ✅ 데이터센터 최적화
- 효과: ~50% 이상 개선

### ⚠️ 주의사항

1. **메모리-속도 트레이드오프**
   - 배치 크기 증가 시 OOM 위험
   - 파이프라인 버퍼 메모리 증가

2. **병렬화 한계**
   - I/O 대역폭 제한
   - 동기화 오버헤드

3. **측정의 중요성**
   - 시뮬레이션과 실제는 다름
   - 실제 프로파일링으로 검증 필수